# Akkadian lemmatizer (BART) - Colab version

Fine-tunes `facebook/bart-base` as a seq2seq lemmatizer. Input: one token's transliteration. Target: `<POS> ... <LEMMA> ... <GLOSS> ...`.

**Before running:** Runtime > Change runtime type > **T4 GPU**. Put this notebook and `train.conllu` in `MyDrive/OB PNs/`.
Edit the config cell (cell 4), then run cells top to bottom. If the session disconnects, re-run everything: training resumes from the last checkpoint saved on Drive.

In [ ]:
!pip install -q polars
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU found. Use Runtime > Change runtime type > T4 GPU, then restart.")

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
WORK_DIR = "/content/drive/MyDrive/OB PNs"
os.chdir(WORK_DIR)
print("Working in:", os.getcwd())
print(os.listdir("."))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working in: /content/drive/MyDrive/OB PNs
['train.conllu', 'lemmatizer_train.ipynb']


In [ ]:
# ---------------- CONFIG (replaces .env / argparse) ----------------
DATA_FILE = "train.conllu"          # relative to the OB PNs folder
OUTPUT_DIR = "lemma_model"          # saved in the OB PNs folder (Drive), so it survives disconnects
MODEL_NAME = "facebook/bart-base"

EPOCHS = 3
BATCH_SIZE = 128                    # .env had 8; sized for an A100 (drop to 64 if you hit out-of-memory)
LEARNING_RATE = 1e-4                # scaled up for the larger batch; use 5e-5 if the loss is unstable
MAX_LENGTH = 64                     # .env had 128; check the truncation report below
SEED = 42

# Column layout, 1-indexed; negative counts from the end (-1 = last column)
FORM_COL = 2
LEMMA_COL = 3
POS_COL = 5
GLOSS_COL = 11

DEDUPLICATE = False                  # drop repeated (form, pos, lemma, gloss) rows; big speedup on 1M+ rows
SAVE_STEPS = 1000                   # checkpoint every N steps (to Drive)
RESUME = True                       # resume from the latest checkpoint in OUTPUT_DIR if one exists
                                    # (set False, or delete the lemma_model folder, when changing settings between runs)
# --------------------------------------------------------------------

# tolerate the "connlu" spelling of the filename
if not os.path.exists(DATA_FILE) and os.path.exists("train.connlu"):
    DATA_FILE = "train.connlu"
print("Using data file:", DATA_FILE, "| exists:", os.path.exists(DATA_FILE))

Using data file: train.conllu | exists: True


In [ ]:
import polars as pl
from datasets import Dataset

POS_TAG = "<POS>"
LEMMA_TAG = "<LEMMA>"
GLOSS_TAG = "<GLOSS>"


def _resolve_column(names, index):
    """Resolve a 1-indexed (possibly negative) column position to its column name."""
    n = len(names)
    pos = index if index > 0 else n + index + 1
    if pos < 1 or pos > n:
        raise ValueError(f"Column index {index} is out of range for a file with {n} columns.")
    return names[pos - 1]


def read_conllu(path, form_col, lemma_col, pos_col, gloss_col):
    """Load a conllu file and build (input_text, target_text) pairs."""
    lazy = pl.scan_csv(
        path,
        has_header=False,
        separator="\t",
        quote_char=None,         # glosses may contain quotes
        comment_prefix="#",      # skip CoNLL-U comment lines
        infer_schema_length=0,   # read every column as string (IDs like "1-2" would break int inference)
    )
    names = lazy.collect_schema().names()

    form_name = _resolve_column(names, form_col)
    lemma_name = _resolve_column(names, lemma_col)
    pos_name = _resolve_column(names, pos_col)
    gloss_name = _resolve_column(names, gloss_col)
    print(f"Detected {len(names)} columns. Mapping -> form: {form_name}, lemma: {lemma_name}, pos: {pos_name}, gloss: {gloss_name}")

    df = (
        lazy.select(
            input_text=pl.col(form_name),
            target_text=pl.concat_str(
                [
                    pl.lit(POS_TAG), pl.col(pos_name),
                    pl.lit(LEMMA_TAG), pl.col(lemma_name),
                    pl.lit(GLOSS_TAG), pl.col(gloss_name),
                ],
                separator=" ",
            ),
        )
        .drop_nulls()
    ).collect()

    if df.is_empty():
        raise ValueError("No rows loaded. Check the column settings in the config cell against this file's layout.")
    return df


def parse_prediction(text):
    """Turn a generated string back into {pos, lemma, gloss}."""
    result = {"pos": None, "lemma": None, "gloss": None}
    for tag, key in ((POS_TAG, "pos"), (LEMMA_TAG, "lemma"), (GLOSS_TAG, "gloss")):
        if tag in text:
            start = text.index(tag) + len(tag)
            end = len(text)
            for other_tag in (POS_TAG, LEMMA_TAG, GLOSS_TAG):
                if other_tag != tag:
                    idx = text.find(other_tag, start)
                    if idx != -1:
                        end = min(end, idx)
            result[key] = text[start:end].strip()
    return result

## Load data and preview the column mapping
Check that the sample rows below look right (form on the left, `<POS> ... <LEMMA> ... <GLOSS> ...` on the right) before training.

In [ ]:
df = read_conllu(DATA_FILE, FORM_COL, LEMMA_COL, POS_COL, GLOSS_COL)
print(f"Loaded {len(df):,} rows.")

if DEDUPLICATE:
    df = df.unique(subset=["input_text", "target_text"], maintain_order=True)
    print(f"After deduplication: {len(df):,} unique rows.")

with pl.Config(fmt_str_lengths=80):
    print(df.head(10))

dataset = Dataset.from_polars(df)

Detected 17 columns. Mapping -> form: column_2, lemma: column_3, pos: column_5, gloss: column_11
Loaded 1,047,105 rows.
shape: (10, 2)
┌────────────┬───────────────────────────────────────┐
│ input_text ┆ target_text                           │
│ ---        ┆ ---                                   │
│ str        ┆ str                                   │
╞════════════╪═══════════════════════════════════════╡
│ x          ┆ <POS> u <LEMMA> _ <GLOSS> _           │
│ GE₆        ┆ <POS> N <LEMMA> mūšu <GLOSS> night    │
│ 28         ┆ <POS> n <LEMMA> _ <GLOSS> _           │
│ ina        ┆ <POS> PRP <LEMMA> ina <GLOSS> in      │
│ ZALAG₂     ┆ <POS> N <LEMMA> nūru <GLOSS> light    │
│ x          ┆ <POS> u <LEMMA> _ <GLOSS> _           │
│ 3          ┆ <POS> n <LEMMA> _ <GLOSS> _           │
│ KUŠ₃       ┆ <POS> N <LEMMA> ammatu <GLOSS> unit   │
│ dele-bat   ┆ <POS> CN <LEMMA> Dilbat <GLOSS> Venus │
│ x          ┆ <POS> u <LEMMA> _ <GLOSS> _           │
└────────────┴──────────────────────────

## Tokenizer, model, and tokenization
Uses dynamic padding (each batch is padded only to its longest example) and masks padding in the labels with -100.

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, set_seed

set_seed(SEED)
os.environ["WANDB_DISABLED"] = "true"

import logging
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)  # hide the harmless "unauthenticated requests" warning

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.add_tokens([POS_TAG, LEMMA_TAG, GLOSS_TAG])
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.resize_token_embeddings(len(tokenizer))

# truncation report on a sample, to sanity-check MAX_LENGTH
sample_df = df.head(20000)
in_lens = [len(x) for x in tokenizer(sample_df["input_text"].to_list())["input_ids"]]
out_lens = [len(x) for x in tokenizer(sample_df["target_text"].to_list())["input_ids"]]
print(f"input  tokens: max={max(in_lens)}, >{MAX_LENGTH}: {sum(l > MAX_LENGTH for l in in_lens)/len(in_lens):.2%}")
print(f"target tokens: max={max(out_lens)}, >{MAX_LENGTH}: {sum(l > MAX_LENGTH for l in out_lens)/len(out_lens):.2%}")


def _tokenize(batch):
    return tokenizer(batch["input_text"], text_target=batch["target_text"], truncation=True, max_length=MAX_LENGTH)


tokenized_dataset = dataset.map(_tokenize, batched=True, remove_columns=["input_text", "target_text"])
print(tokenized_dataset)

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


input  tokens: max=35, >64: 0.00%
target tokens: max=38, >64: 0.00%


Map:   0%|          | 0/1047105 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1047105
})


## Train
Checkpoints go to `OB PNs/lemma_model/` on Drive every `SAVE_STEPS` steps (last 2 kept). If the session dies, re-run the notebook from the top and it resumes automatically.

In [ ]:
import inspect
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint

# precision: bf16 + tf32 on Ampere or newer (A100, L4); fp16 on older GPUs (T4); full precision on CPU
cuda = torch.cuda.is_available()
use_bf16 = cuda and torch.cuda.is_bf16_supported()
use_fp16 = cuda and not use_bf16
use_tf32 = cuda and torch.cuda.get_device_capability()[0] >= 8
print(f"bf16={use_bf16}, fp16={use_fp16}, tf32={use_tf32}")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    bf16=use_bf16,
    fp16=use_fp16,
    tf32=use_tf32,
    dataloader_num_workers=2,
    optim="adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch",
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    logging_steps=200,
    report_to="none",
    seed=SEED,
)

collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100)

# newer transformers renamed Trainer(tokenizer=...) to processing_class=...
tok_kwarg = "processing_class" if "processing_class" in inspect.signature(Trainer.__init__).parameters else "tokenizer"
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=collator,
    **{tok_kwarg: tokenizer},
)

last_ckpt = get_last_checkpoint(OUTPUT_DIR) if (RESUME and os.path.isdir(OUTPUT_DIR)) else None
print("Resuming from:", last_ckpt if last_ckpt else "scratch")
trainer.train(resume_from_checkpoint=last_ckpt)

bf16=True, fp16=False, tf32=True
Resuming from: scratch


Step,Training Loss
200,1.599449
400,0.418546
600,0.314991
800,0.265826
1000,0.231317
1200,0.206235
1400,0.190671
1600,0.173582
1800,0.155990
2000,0.151231


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=24543, training_loss=0.08398981156143433, metrics={'train_runtime': 1803.6361, 'train_samples_per_second': 1741.657, 'train_steps_per_second': 13.608, 'total_flos': 4.265940863987712e+16, 'train_loss': 0.08398981156143433, 'epoch': 3.0})

In [ ]:
print(f"Saving final model to {os.path.join(WORK_DIR, OUTPUT_DIR)} ...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved.")

Saving final model to /content/drive/MyDrive/OB PNs/lemma_model ...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved.


## Sanity check

In [ ]:
model.eval()
for form in df["input_text"].head(5).to_list():
    inputs = tokenizer(form, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_length=MAX_LENGTH)
    decoded = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"{form}\n  -> {decoded}\n  -> {parse_prediction(decoded)}\n")

x
  -> <POS> u <LEMMA> _ <GLOSS> _
  -> {'pos': 'u', 'lemma': '_', 'gloss': '_'}

GE₆
  -> <POS> N <LEMMA> mūšu <GLOSS> night
  -> {'pos': 'N', 'lemma': 'mūšu', 'gloss': 'night'}

28
  -> <POS> n <LEMMA> _ <GLOSS> _
  -> {'pos': 'n', 'lemma': '_', 'gloss': '_'}

ina
  -> <POS> PRP <LEMMA> ina <GLOSS> in
  -> {'pos': 'PRP', 'lemma': 'ina', 'gloss': 'in'}

ZALAG₂
  -> <POS> N <LEMMA> nūru <GLOSS> light
  -> {'pos': 'N', 'lemma': 'nūru', 'gloss': 'light'}

